In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import MBart50Tokenizer, MBartForConditionalGeneration, Trainer, TrainingArguments

# **Reading file and displaying the first 5 rows in the df**

In [ ]:
file_path = "/content/sample_data/TamiSpell_Isolated_SE.csv"
df = pd.read_csv(file_path)

In [ ]:
df.head()

,சென்னை தமிழ் சங்கம்,சென்னைத் தமிழ்ச் சங்கம்
0,மதுரை தமிழ் சங்கம்,மதுரைத் தமிழ்ச் சங்கம்
1,புது கல்வி திட்டம்,புதுக் கல்வித் திட்டம்
2,குடும்பகளை காப்பாற்ற,குடும்பகளைக் காப்பாற்ற
3,அந்த செடி,அந்தச் செடி
4,இந்த குழந்தை,இந்தக் குழந்தை


# **Helpers..**

In [ ]:
def load_data(file_path, tokenizer, max_length=128, test_size=0.2):
    data = pd.read_csv(file_path)
    train_data, val_data = train_test_split(data, test_size=test_size)
    return Seq2SeqDataset(train_data, tokenizer, max_length), Seq2SeqDataset(val_data, tokenizer, max_length)

In [ ]:
class Seq2SeqDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        source_text = str(self.data.iloc[idx, 0])  # Error sentence
        target_text = str(self.data.iloc[idx, 1])  # Correct sentence

        source = self.tokenizer(source_text, padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
        target = self.tokenizer(target_text, padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")

        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": target["input_ids"].squeeze()
        }

# **Loading the model**

In [ ]:
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer.src_lang = "ta_IN"
tokenizer.tgt_lang = "ta_IN"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

In [ ]:
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

# **Loading data**

In [ ]:
train_dataset, val_dataset = load_data("/content/sample_data/TamiSpell_Isolated_SE.csv", tokenizer)

# **Training...**

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/sample_data/BestMbartSandhi",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=3e-5,
    save_total_limit=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/content/sample_data/logs",
    logging_steps=250,
    fp16=True if torch.cuda.is_available() else False,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
torch.cuda.is_available()

True

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.013749
2,1.611288,0.010430
3,0.004523,0.010219


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=501, training_loss=0.8062955295906854, metrics={'train_runtime': 1043.5853, 'train_samples_per_second': 1.915, 'train_steps_per_second': 0.48, 'total_flos': 541240659542016.0, 'train_loss': 0.8062955295906854, 'epoch': 3.0})

In [ ]:
model.save_pretrained("BestmbartmodelSANDHI")
tokenizer.save_pretrained("BestmbartTokenizerSANDHI")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('BestmbartTokenizerSANDHI/tokenizer_config.json',
 'BestmbartTokenizerSANDHI/tokenizer.json')

In [ ]:
from evaluate import load

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
from evaluate import load

In [ ]:
tokenizer = MBart50Tokenizer.from_pretrained("BestmbartTokenizerSANDHI")
model = MBartForConditionalGeneration.from_pretrained("BestmbartmodelSANDHI")

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

In [ ]:
data = pd.read_csv("/content/sample_data/TamiSpell_Isolated_SE.csv")  # Load same dataset
_, val_data = train_test_split(data, test_size=0.2)

In [ ]:
val_data_sample = val_data

In [ ]:
def generate_predictions(model, tokenizer, input_texts):
    model.eval()
    predictions = []

    for text in input_texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            output_tokens = model.generate(**inputs, max_length=128)
        pred_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        predictions.append(pred_text)

    return predictions

In [ ]:
val_error_sentences = val_data_sample.iloc[:, 0].tolist()  # Error sentences
val_correct_sentences = val_data_sample.iloc[:, 1].tolist()  # Correct sentences

In [ ]:
predicted_sentences = generate_predictions(model, tokenizer, val_error_sentences)

In [ ]:
bleu_metric = load("bleu")

In [ ]:
# Convert to required format
predictions_for_bleu = [pred for pred in predicted_sentences]  # Remove nested lists
references_for_bleu = [[ref] for ref in val_correct_sentences]  # Ensure correct format

In [ ]:
# Compute BLEU Score
bleu_score = bleu_metric.compute(predictions=predictions_for_bleu, references=references_for_bleu)

In [ ]:
# Compute Exact Match Accuracy
exact_matches = sum([1 for p, t in zip(predicted_sentences, val_correct_sentences) if p.strip() == t.strip()])
accuracy = exact_matches / len(val_correct_sentences)

AttributeError: 'float' object has no attribute 'strip'

In [ ]:
val_error_sentences = (
    val_data_sample.iloc[:, 0]
    .fillna("")
    .astype(str)
    .tolist()
)

In [ ]:
val_correct_sentences = (
    val_data_sample.iloc[:, 1]
    .fillna("")
    .astype(str)
    .tolist()
)

In [ ]:
def generate_predictions(model, tokenizer, input_texts, max_length=128):
    model.eval()
    device = next(model.parameters()).device
    predictions = []

    for text in input_texts:
        inputs = tokenizer(
            str(text),
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            output_tokens = model.generate(**inputs, max_length=max_length)

        pred_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        predictions.append(pred_text)

    return predictions

In [ ]:
predicted_sentences = [str(s).strip() for s in predicted_sentences]
val_correct_sentences = [str(s).strip() for s in val_correct_sentences]

In [ ]:
exact_matches = sum(1 for p, t in zip(predicted_sentences, val_correct_sentences) if p == t)
accuracy = exact_matches / len(val_correct_sentences)


In [ ]:
print("Exact match accuracy:", accuracy)

Exact match accuracy: 0.9341317365269461


In [ ]:
predictions_for_bleu = [str(p) for p in predicted_sentences]
references_for_bleu = [[str(r)] for r in val_correct_sentences]

In [ ]:
bleu_score = bleu_metric.compute(predictions=predictions_for_bleu, references=references_for_bleu)
print(bleu_score)

{'bleu': 0.9558474361186502, 'precisions': [0.9645776566757494, 0.955, 0.9565217391304348, 0.9473684210526315], 'brevity_penalty': 1.0, 'length_ratio': 1.0138121546961325, 'translation_length': 367, 'reference_length': 362}


In [ ]:
# Print results
print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")
print(f"BLEU Score: {bleu_score['bleu'] * 100:.2f}%")

Exact Match Accuracy: 93.41%
BLEU Score: 95.58%
